# House Prices: previsão de preço de venda de imóveis

Outro clássico do Kaggle (*House Prices - Advanced Regression Techniques*,
Ames, Iowa). Uso aqui uma amostra de 128 imóveis do dataset original
(1.460 linhas, 81 colunas) - o suficiente para demonstrar o pipeline de
regressão de ponta a ponta sem deixar o notebook pesado; os números
absolutos de erro tendem a ficar mais instáveis do que rodando na base
completa, e isso está sinalizado nas conclusões.

**Pergunta de negócio:** dado um conjunto de características físicas e de
localização do imóvel, qual o preço de venda esperado?

Dataset: `data/house_prices_sample.csv`.

## Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
RANDOM_STATE = 42

## Carregando os dados

In [ ]:
df = pd.read_csv("data/house_prices_sample.csv")
print(df.shape)
df[["OverallQual", "GrLivArea", "YearBuilt", "Neighborhood", "SalePrice"]].head()

## Análise exploratória

`SalePrice` é a variável alvo. Preços de imóveis costumam ter distribuição
assimétrica à direita (poucos imóveis muito caros puxam a cauda) - vale
conferir antes de escolher a métrica de erro e decidir se compensa
trabalhar em log.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("Distribuição de SalePrice")

sns.histplot(np.log1p(df["SalePrice"]), kde=True, ax=axes[1])
axes[1].set_title("Distribuição de log(SalePrice + 1)")

plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
correlations.head(10)

In [ ]:
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", hue="OverallQual", palette="viridis")
plt.title("Área habitável x Preço de venda, colorido por qualidade geral do imóvel")
plt.show()

## Feature engineering e tratamento de nulos

Nesse dataset a maior parte dos "nulos" em colunas categóricas na verdade
significa "o imóvel não tem essa característica" (`PoolQC` nulo = sem
piscina, `Alley` nulo = sem acesso por viela) - não é dado faltando de
verdade. Por isso o tratamento aqui é diferente de um imputer genérico:
primeiro eu preencho esses casos explicitamente, e só then deixo o
`SimpleImputer` cuidar do que sobrar.

In [ ]:
cols_na_means_none = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
for col in cols_na_means_none:
    if col in df.columns:
        df[col] = df[col].fillna("None")

# variável derivada: idade do imóvel na venda, e se passou por reforma
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["WasRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)

In [ ]:
numeric_features = [
    "OverallQual", "OverallCond", "GrLivArea", "TotalBsmtSF", "1stFlrSF",
    "GarageCars", "GarageArea", "FullBath", "TotRmsAbvGrd", "HouseAge",
    "WasRemodeled",
]
categorical_features = ["Neighborhood", "BldgType", "HouseStyle", "KitchenQual"]

X = df[numeric_features + categorical_features]
y = np.log1p(df["SalePrice"])  # regressão no log do preço

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

## Comparando modelos

In [ ]:
models = {
    "linear_regression": LinearRegression(),
    "ridge": Ridge(alpha=10.0, random_state=RANDOM_STATE),
    "random_forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
}

for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
    rmse = -scores
    print(f"{name}: RMSE médio (log-price, 5-fold CV) = {rmse.mean():.4f} +/- {rmse.std():.4f}")

## Avaliação no conjunto de teste

Random Forest costuma vencer Linear/Ridge nesse tipo de problema por
capturar não-linearidades (ex: o efeito de `OverallQual` no preço não é
constante ao longo da escala), mas com só ~100 imóveis de treino o
resultado pode oscilar bastante entre execuções - por isso a validação
cruzada acima importa mais que um único número de teste.

In [ ]:
final_pipe = Pipeline([("preprocess", preprocess), ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE))])
final_pipe.fit(X_train, y_train)

y_pred_log = final_pipe.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))  # sqrt manual: compatível com qualquer versão do sklearn
mae = mean_absolute_error(y_test_real, y_pred)
r2 = r2_score(y_test_real, y_pred)

print(f"RMSE (R$): {rmse:,.0f}")
print(f"MAE (R$): {mae:,.0f}")
print(f"R²: {r2:.3f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test_real, y_pred, alpha=0.6)
lims = [y_test_real.min(), y_test_real.max()]
plt.plot(lims, lims, "r--", label="previsão perfeita")
plt.xlabel("Preço real")
plt.ylabel("Preço previsto")
plt.title("Preço real x previsto - conjunto de teste")
plt.legend()
plt.show()

In [ ]:
importances = final_pipe.named_steps["model"].feature_importances_
feature_names = final_pipe.named_steps["preprocess"].get_feature_names_out()

top_features = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)
top_features.plot(kind="barh", figsize=(8, 6))
plt.title("Importância das features - Random Forest")
plt.gca().invert_yaxis()
plt.show()

## Conclusões e limitações

`OverallQual` (qualidade geral do acabamento) e `GrLivArea` (área
habitável) são historicamente, em qualquer corte desse dataset, os
preditores mais fortes de preço - o que já aparece na correlação simples
antes de qualquer modelo. O ganho de um Random Forest sobre regressão
linear normalmente vem de capturar como esse efeito muda dependendo do
bairro e do padrão construtivo.

Limitação relevante: esse notebook roda sobre uma amostra de 128 imóveis
(a competição original tem 1.460). Para uso real eu rodaria com a base
completa via Kaggle - o pipeline (feature engineering, tratamento de nulos
categóricos, comparação de modelos, avaliação) é o mesmo, só troca o
`pd.read_csv`.